# Several metrics in one pass

Practice notebook for the datafromzero SQL lesson **Several metrics in one pass**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. One conditional column

Grain: one row per store.

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(*) AS orders,
       SUM(CASE WHEN payment_method = 'card' THEN 1 ELSE 0 END) AS card
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 2. The other two columns, with the other form

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(*) AS orders,
       SUM(CASE WHEN payment_method = 'card' THEN 1 ELSE 0 END) AS card,
       COUNT(CASE WHEN payment_method = 'mobile' THEN 1 END) AS mobile,
       COUNT(CASE WHEN payment_method = 'cash' THEN 1 END) AS cash
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 3. A percentage, with its denominator named

The part is the store's mobile orders. The whole is all orders of that store, canceled included.

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(*) AS orders,
       COUNT(CASE WHEN payment_method = 'mobile' THEN 1 END) AS mobile,
       ROUND(100.0 * COUNT(CASE WHEN payment_method = 'mobile' THEN 1 END) / COUNT(*), 1) AS mobile_pct
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 4. Divide safely with `NULLIF`

Harbor opened on April 1, so it has no orders in the first quarter. Run this and read Harbor's last column.

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(CASE WHEN ordered_at < '2026-04-01' THEN 1 END) AS q1_orders,
       COUNT(CASE WHEN ordered_at < '2026-04-01' AND payment_method = 'mobile' THEN 1 END) AS q1_mobile,
       ROUND(100.0 * COUNT(CASE WHEN ordered_at < '2026-04-01' AND payment_method = 'mobile' THEN 1 END)
             / COUNT(CASE WHEN ordered_at < '2026-04-01' THEN 1 END), 1) AS q1_mobile_pct
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

Now wrap the denominator in `NULLIF(..., 0)`.

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(CASE WHEN ordered_at < '2026-04-01' THEN 1 END) AS q1_orders,
       COUNT(CASE WHEN ordered_at < '2026-04-01' AND payment_method = 'mobile' THEN 1 END) AS q1_mobile,
       ROUND(100.0 * COUNT(CASE WHEN ordered_at < '2026-04-01' AND payment_method = 'mobile' THEN 1 END)
             / NULLIF(COUNT(CASE WHEN ordered_at < '2026-04-01' THEN 1 END), 0), 1) AS q1_mobile_pct
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 5. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders,
       SUM(CASE WHEN payment_method = 'card' THEN 1 ELSE 0 END) AS card,
       COUNT(CASE WHEN payment_method = 'mobile' THEN 1 END) AS mobile,
       COUNT(CASE WHEN payment_method = 'cash' THEN 1 END) AS cash
FROM orders
""").show()

## Exercise 1. Orders in the first and the second quarter

The first lines are given. Add the second column for orders on or after April 1, name the table, and group and sort by
store.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    SELECT store_id,
           COUNT(CASE WHEN ordered_at < '2026-04-01' THEN 1 END) AS q1_orders,
    """).show()
except Exception as e:
    print(e)


## Exercise 2. Cash share, two denominators

Before you run anything, write two sentences in the first cell, one per percentage: what is the part, and what is the
whole? Then, for each store, show the share of its orders paid in cash (`cash_pct_of_all`) and the share of its
completed orders paid in cash (`cash_pct_of_completed`), each rounded to one decimal.

In [ ]:
# My expected result, in words:
#

In [ ]:
# My query:

## Exercise 3. Spot the flaw

A colleague builds the card and mobile columns with `COUNT` and an `ELSE 0`. The query runs without an error. Run it
and compare each column with the store's order count: what do you notice, and why? Then write a corrected query.

In [ ]:
con.sql("""
SELECT store_id,
       COUNT(CASE WHEN payment_method = 'card' THEN 1 ELSE 0 END) AS card,
       COUNT(CASE WHEN payment_method = 'mobile' THEN 1 ELSE 0 END) AS mobile
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

In [ ]:
# Your corrected query: